# Flow matching for simple distributions

## Prepare toy data

Generate target distribution data (two gaussians) and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

source_data = np.random.randn(1000, 2)

target_data = np.vstack([
    np.random.multivariate_normal([5, 5], np.array([[1, -0.75], [-0.75, 1]]), 500),
    np.random.multivariate_normal([-5, -5], np.array([[1, -0.75], [-0.75, 1]]), 500)
])

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

fig.update_layout(
    title="Source vs Target Distributions",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[-10, 10], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig.show()

## Basic flow matching

Example of independent coupling between source and target data points

In [ ]:
n_pairs = 1000

src_idx = np.random.randint(0, source_data.shape[0], size=n_pairs)
tgt_idx = np.random.randint(0, target_data.shape[0], size=n_pairs)

couplings = [(source_data[i], target_data[j]) for i, j in zip(src_idx, tgt_idx)]

Visualize the generated couplings

In [ ]:
x_lines, y_lines = [], []
for src, tgt in couplings:
    x_lines.extend([src[0], tgt[0], None])
    y_lines.extend([src[1], tgt[1], None])

fig_pairs = go.Figure()

fig_pairs.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig_pairs.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

fig_pairs.add_trace(
    go.Scatter(
        x=x_lines,
        y=y_lines,
        mode="lines",
        name="Couplings",
        line=dict(color="rgba(120,120,120,0.25)", width=1),
        hoverinfo="skip",
    )
)

fig_pairs.update_layout(
    title="Independent Couplings",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[-10, 10], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig_pairs.show()

Define a simple MLP with ReLU activations as the velocity field estimation network.

In [ ]:
import torch
import torch.nn as nn

class FlowMLP(nn.Module):
    def __init__(self, input_output_dim: int, hidden_dim: int, num_blocks: int):
        super().__init__()
        if num_blocks < 1:
            raise ValueError("num_blocks must be >= 1")

        layers = []
        in_dim = input_output_dim

        for _ in range(num_blocks):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        layers.append(nn.Linear(hidden_dim, input_output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

Train the network through backpropagation

In [ ]:
import torch.optim as optim

# Prepare training tensors from existing couplings
src_tensor = torch.tensor(source_data, dtype=torch.float32)
tgt_tensor = torch.tensor(target_data, dtype=torch.float32)

# Instantiate model for 2D data
model = FlowMLP(input_output_dim=source_data.shape[1], hidden_dim=128, num_blocks=3)

# Training setup
batch_size = 128
num_updates = 10000
learning_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=num_updates)
criterion = nn.MSELoss()

# Train
model.train()
ema_loss = 0.0
for update in range(num_updates):
    # Create minibatch of independent couplings of source-target points
    src_batch = src_tensor[np.random.randint(0, src_tensor.shape[0], size=batch_size), :]
    tgt_batch = tgt_tensor[np.random.randint(0, tgt_tensor.shape[0], size=batch_size), :]

    # Sample t ~ Uniform[0, 1] for each pair in minibatch
    t = torch.rand(src_batch.shape[0], 1)

    # Linear interpolation x_t = (1-t) * src + t * tgt
    x_t = (1.0 - t) * src_batch + t * tgt_batch

    # Target velocity vector: tgt - src
    v_target = tgt_batch - src_batch

    # Predict and optimize
    v_pred = model(x_t)
    loss = criterion(v_pred, v_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ema_loss = 0.99 * ema_loss + 0.01 * loss.item()

    scheduler.step()
    if (update + 1) % 100 == 0 or update == 0:
        print(f"Update {update + 1}/{num_updates} - EMA Loss: {ema_loss:.6f}, lr: {scheduler.get_last_lr()[0]:.6f}")

Visualize the velocity field we have learned

In [ ]:
import plotly.figure_factory as ff

# Evaluate learned velocity field on a 2D grid and visualize vectors
x_min, x_max = -10, 10
y_min, y_max = -10, 10
n_grid = 25
scale = 0.05  # arrow length scaling for visualization

xg = np.linspace(x_min, x_max, n_grid)
yg = np.linspace(y_min, y_max, n_grid)
X, Y = np.meshgrid(xg, yg)
grid_points = np.stack([X.ravel(), Y.ravel()], axis=1)

device = next(model.parameters()).device
model.eval()
with torch.no_grad():
    v_grid = model(torch.tensor(grid_points, dtype=torch.float32, device=device)).cpu().numpy()

# Build line segments for arrows: (x, y) -> (x + scale*vx, y + scale*vy)
x_lines_field, y_lines_field = [], []
for (x0, y0), (vx, vy) in zip(grid_points, v_grid):
    x_lines_field.extend([x0, x0 + scale * vx, None])
    y_lines_field.extend([y0, y0 + scale * vy, None])

fig_field = go.Figure()

quiver = ff.create_quiver(
    grid_points[:, 0],
    grid_points[:, 1],
    scale * v_grid[:, 0],
    scale * v_grid[:, 1],
    scale=1.0,
    arrow_scale=0.35,  # pointy arrow heads
    line=dict(color="rgba(20,20,20,0.55)", width=1),
    name="Predicted velocity vectors",
)

fig_field.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig_field.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

for trace in quiver.data:
    fig_field.add_trace(trace)

fig_field.update_layout(
    title="Learned Velocity Field",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[x_min, x_max], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig_field.show()

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow.

In [ ]:
def euler_integrate(initial_point, velocity_fn, n_steps):
    """
    Euler integration from t=0 to t=1.

    Args:
        initial_point: array-like, shape (d,)
        velocity_fn: callable(point) -> velocity
        n_steps: int, number of integration steps
    
    Returns:
        List of (t, point) tuples, including t=0 and t=1.
    """
    if n_steps < 1:
        raise ValueError("n_steps must be >= 1")

    x = np.asarray(initial_point, dtype=np.float32).copy()
    dt = 1.0 / n_steps

    path = [(0.0, x.copy())]

    for k in range(n_steps):
        v = velocity_fn(x)
        v = np.asarray(v, dtype=np.float32)
        x = x + dt * v
        path.append(((k + 1) * dt, x.copy()))

    return path

Using this integration method we compute the induced trajectories for the original (noisy) data points.

In [ ]:
# Build velocity function from the trained model
model.eval()

def velocity_fn(point):
    with torch.no_grad():
        x = torch.tensor(point, dtype=torch.float32, device=device).unsqueeze(0)
        v = model(x).squeeze(0).cpu().numpy()
    return v

# Integrate trajectories for all original noisy points
n_steps = 50
trajectories = [euler_integrate(p, velocity_fn, n_steps) for p in source_data]

Now we can visualize the generated trajectories

In [ ]:
# Convert trajectories to Plotly line format
x_traj_lines, y_traj_lines = [], []
for traj in trajectories:
    for _, p in traj:
        x_traj_lines.append(p[0])
        y_traj_lines.append(p[1])
    x_traj_lines.append(None)
    y_traj_lines.append(None)

# Collect trajectory end points
end_points = np.stack([traj[-1][1] for traj in trajectories])

# Plot: original points + trajectories + end points
fig_traj = go.Figure()

fig_traj.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Original (source) points",
        marker=dict(color="blue", size=5, opacity=0.7),
    )
)

fig_traj.add_trace(
    go.Scatter(
        x=x_traj_lines,
        y=y_traj_lines,
        mode="lines",
        name="Induced trajectories",
        line=dict(color="rgba(80,80,80,0.25)", width=1),
        hoverinfo="skip",
    )
)

fig_traj.add_trace(
    go.Scatter(
        x=end_points[:, 0],
        y=end_points[:, 1],
        mode="markers",
        name="Trajectory end points",
        marker=dict(color="red", size=5, opacity=0.8),
    )
)

fig_traj.update_layout(
    title="Induced Trajectories from Source Points",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[-10, 10], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=35, b=0, pad=0),
    template="plotly_white",
)

fig_traj.show()

In [ ]:
# Build trajectory array: shape (n_points, n_time, 2)
traj_array = np.stack([[p for _, p in traj] for traj in trajectories])
n_points, n_time, _ = traj_array.shape

def build_history_lines(step_idx):
    x_hist, y_hist = [], []
    for i in range(n_points):
        x_hist.extend(traj_array[i, : step_idx + 1, 0].tolist())
        y_hist.extend(traj_array[i, : step_idx + 1, 1].tolist())
        x_hist.append(None)
        y_hist.append(None)
    return x_hist, y_hist

# Initial frame (t=0): only original points, no path yet
x0 = traj_array[:, 0, 0]
y0 = traj_array[:, 0, 1]

fig_anim = go.Figure(
    data=[
        go.Scatter(
            x=[],
            y=[],
            mode="lines",
            name="Trajectory",
            line=dict(color="rgba(80,80,80,0.25)", width=1),
            hoverinfo="skip",
        ),
        go.Scatter(
            x=x0,
            y=y0,
            mode="markers",
            name="Generated points",
            marker=dict(color="red", size=5, opacity=0.8),
        ),
    ]
)

# Frames for t=1..end
frames = []
for k in range(1, n_time):
    x_hist, y_hist = build_history_lines(k)
    frames.append(
        go.Frame(
            name=str(k),
            data=[
                go.Scatter(x=x_hist, y=y_hist),
                go.Scatter(x=traj_array[:, k, 0], y=traj_array[:, k, 1]),
            ],
        )
    )

fig_anim.frames = frames

# Slider + play controls
slider_steps = [
    dict(
        method="animate",
        args=[
            [str(k)],
            dict(mode="immediate", frame=dict(duration=50, redraw=True), transition=dict(duration=0)),
        ],
        label=str(k),
    )
    for k in range(1, n_time)
]

fig_anim.update_layout(
    title="Animated Induced Trajectories",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[x_min, x_max], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", range=[y_min, y_max], scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=50, b=20, pad=0),
    template="plotly_white",
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.01,
            y=1.00,
            xanchor="left",
            yanchor="top",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=25, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                )
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=45),
            steps=slider_steps,
        )
    ],
)

fig_anim.show()